# Module 4 · Wiring Up the Agent (45 min)

The loop is **complete and read-only**. You supply three pieces in `ha_workshop/agent_config.py`:
the tools to register (**TODO-1**), the observation step (**TODO-2**), and the system prompt
(**TODO-3**). Read the loop, run the baseline, then fill the three TODOs.

In [ ]:
# Make the top-level ha_workshop package importable and anchor relative paths to the repo root.
import os, sys
from pathlib import Path
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'healthagent').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
print('repo root:', ROOT)

In [ ]:
import inspect
from healthagent.agent.loop import run_agent
print(inspect.getsource(run_agent))

### C2 · The shipped agent already runs (out of the box)
Only `compare_window` is wired, so it answers the **activity** question — but **not** the sleep
question yet. Making the sleep question work is your job (TODO-1).

In [ ]:
from healthagent.llm.client import get_client
from healthagent.agent.baseline import run_baseline_agent
client = get_client('auto')
print('ACTIVITY:', run_baseline_agent('How does my activity compare to my goal?', client=client).text)
print('\nSLEEP   :', run_baseline_agent('Why have I been sleeping poorly this week?', client=client).text)

In [ ]:
from ha_workshop import workshop_paths
print('Edit this file for Module 4:')
print('   ', workshop_paths()['agent_config'])

### Your three edits (in `ha_workshop/agent_config.py`)
- **TODO-1** `WORKSHOP_TOOLS = [retrieval.query_sleep, student_tools.query_screen_time, student_tools.detect_change, visualization.plot_timeseries]`
- **TODO-2** in `observe`, return `client.serialize_tool_result(tool_call.id, result)`
- **TODO-3** `SYSTEM_PROMPT = BASE_SYSTEM + '\n' + GROUNDING_CLAUSE + '\n' + REFUSAL_CLAUSE`

Save, then run the cell below (it reloads your config). Before the edits the agent answers
*"I could not gather enough evidence"* — that's the loop failing **safely**, not making things up.

In [ ]:
from ha_workshop import reload_ha_workshop
_, cfg = reload_ha_workshop()
for q in ['How does my activity compare to my goal?',
          'Why have I been sleeping poorly this week?']:
    ans = cfg.run(q, client=client)
    print('\nQ:', q)
    print('  tools used:', [t for s in ans.trace for t in s['tool_calls']])
    print('  grounded:', ans.grounded)
    print('  answer:', ans.text[:320])

In [ ]:
# Safety check: a medical-advice probe should be refused — but only after you add the
# [REFUSAL] clause in TODO-3.
print(cfg.run('Should I take melatonin for my sleep?', client=client).text)

### Discussion (no code): single agent vs. orchestrator-with-specialists
```
        ┌──────────── single agent (this tutorial / Part 1) ───────────┐
  user → plan → pick tool → run → observe → … → grounded answer

        ┌──── multi-agent orchestrator (Part 2 / GLOSS) ────┐
  user → planner ─┬─ retrieval agent ─┐
                  ├─ analysis agent  ─┼─ synthesizer → answer
                  └─ viz agent       ─┘
```
Multi-agent buys parallelism and specialization at the cost of coordination, latency, and more
failure modes — the focus of the companion GLOSS tutorial.